In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

print("Libraries imported successfully!")

In [ ]:
from google.colab import files

uploaded = files.upload()

image_name = list(uploaded.keys())[0]

image = cv2.imread(image_name)

if image is None:
    raise ValueError("Image could not be loaded.")

print("Image loaded successfully:", image_name)
print("Image shape:", image.shape)

In [ ]:
# Convert BGR to RGB for Matplotlib
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(8, 6))
plt.imshow(gray, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")
plt.show()

In [ ]:
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

plt.figure(figsize=(8, 6))
plt.imshow(blurred, cmap="gray")
plt.title("Gaussian Blurred Image")
plt.axis("off")
plt.show()

In [ ]:
# Global Thresholding

threshold_value = 127

_, global_threshold = cv2.threshold(
    blurred,
    threshold_value,
    255,
    cv2.THRESH_BINARY
)

plt.figure(figsize=(8, 6))
plt.imshow(global_threshold, cmap="gray")
plt.title("Global Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Otsu's Thresholding

otsu_threshold, otsu_result = cv2.threshold(
    blurred,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Optimal Otsu Threshold:", otsu_threshold)

plt.figure(figsize=(8, 6))
plt.imshow(otsu_result, cmap="gray")
plt.title("Otsu's Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Adaptive Mean Thresholding

adaptive_mean = cv2.adaptiveThreshold(
    blurred,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

plt.figure(figsize=(8, 6))
plt.imshow(adaptive_mean, cmap="gray")
plt.title("Adaptive Mean Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Adaptive Gaussian Thresholding

adaptive_gaussian = cv2.adaptiveThreshold(
    blurred,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,
    2
)

plt.figure(figsize=(8, 6))
plt.imshow(adaptive_gaussian, cmap="gray")
plt.title("Adaptive Gaussian Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Use Otsu result as the starting binary image

binary = otsu_result.copy()

# Make sure foreground is white
if np.mean(binary == 255) < 0.5:
    binary = cv2.bitwise_not(binary)

plt.figure(figsize=(8, 6))
plt.imshow(binary, cmap="gray")
plt.title("Binary Image for Watershed")
plt.axis("off")
plt.show()

In [ ]:
# Remove noise and obtain sure foreground/background

kernel = np.ones((3, 3), np.uint8)

opening = cv2.morphologyEx(
    binary,
    cv2.MORPH_OPEN,
    kernel,
    iterations=2
)

sure_bg = cv2.dilate(
    opening,
    kernel,
    iterations=3
)

dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

_, sure_fg = cv2.threshold(
    dist_transform,
    0.5 * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(sure_fg)

unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

plt.figure(figsize=(8, 6))
plt.imshow(unknown, cmap="gray")
plt.title("Unknown Region")
plt.axis("off")
plt.show()

In [ ]:
# Marker labeling

num_labels, markers = cv2.connectedComponents(sure_fg)

# Add one so that background becomes 1 instead of 0
markers = markers + 1

# Mark unknown region as 0
markers[unknown == 255] = 0

# Apply watershed
watershed_image = image.copy()

markers = cv2.watershed(
    watershed_image,
    markers
)

# Boundary pixels are marked as -1
watershed_image[markers == -1] = [0, 0, 255]

watershed_rgb = cv2.cvtColor(
    watershed_image,
    cv2.COLOR_BGR2RGB
)

plt.figure(figsize=(8, 6))
plt.imshow(watershed_rgb)
plt.title("Watershed Segmentation")
plt.axis("off")
plt.show()

In [ ]:
# Reshape image into a list of pixels

pixels = image_rgb.reshape(
    (-1, 3)
)

pixels = np.float32(pixels)

print("Total pixels:", pixels.shape[0])
print("Features per pixel:", pixels.shape[1])

In [ ]:
# Number of color clusters

K = 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(pixels)

centers = np.uint8(
    kmeans.cluster_centers_
)

segmented_pixels = centers[labels]

segmented_image = segmented_pixels.reshape(
    image_rgb.shape
)

plt.figure(figsize=(8, 6))
plt.imshow(segmented_image)
plt.title(f"K-Means Image Segmentation (K={K})")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))

plt.subplot(2, 3, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(global_threshold, cmap="gray")
plt.title("Global Thresholding")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(otsu_result, cmap="gray")
plt.title("Otsu's Thresholding")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(adaptive_gaussian, cmap="gray")
plt.title("Adaptive Thresholding")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(watershed_rgb)
plt.title("Watershed")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(segmented_image)
plt.title("K-Means")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    gray.ravel(),
    bins=256,
    range=(0, 256)
)

plt.title("Grayscale Image Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Calculate simple foreground percentages

methods = {
    "Global Threshold": global_threshold,
    "Otsu": otsu_result,
    "Adaptive": adaptive_gaussian
}

for name, result in methods.items():

    foreground_pixels = np.sum(result == 255)
    total_pixels = result.size

    percentage = (
        foreground_pixels / total_pixels
    ) * 100

    print(
        f"{name}: "
        f"{percentage:.2f}% foreground pixels"
    )

In [ ]:
print("""
FINAL OBSERVATION

1. Global Thresholding uses a fixed threshold and works well when
   foreground and background have clearly different intensities.

2. Otsu's Thresholding automatically determines a suitable threshold
   from the image histogram.

3. Adaptive Thresholding is useful when illumination is uneven because
   it calculates thresholds for local regions.

4. Watershed Segmentation is useful for separating touching or
   overlapping objects.

5. K-Means performs color-based segmentation by grouping similar
   pixels into different clusters.

The most suitable segmentation method depends on the image,
illumination conditions, object boundaries, background complexity,
and required accuracy.
""")